# Audio chain test

Runs a WAV file through the same audio processing stages as the firmware:

1. Resample input to 20 kHz (firmware DSP rate)
2. High-pass Chebyshev I biquad, 200 Hz (Stage 3b)
3. Voice compressor, 4:1 ratio (Stage 3c)
4. Low-pass Chebyshev I biquad, 2800 Hz (Stage 3d)
5. Write processed WAV and display spectrograms

The filter coefficients and compressor parameters are copied verbatim from `src/dsp.rs`
and `tools/design_audio_filters.ipynb` so that the Python output exactly matches
what the firmware produces.

In [ ]:
import numpy as np
import scipy.signal as signal
import scipy.io.wavfile as wavfile
import matplotlib.pyplot as plt
from pathlib import Path

## Parameters

Set `INPUT_WAV` to any mono or stereo WAV file.  The notebook resamples it to
`FS = 20 000 Hz` before processing.

In [ ]:
INPUT_WAV  = 'input.wav'   # path to input WAV (any sample rate, mono or stereo)
OUTPUT_WAV = 'output.wav'  # processed output at 20 kHz

FS = 20_000   # firmware DSP rate [Hz]

## Filter coefficients

Copied verbatim from `src/dsp.rs` (generated by `tools/design_audio_filters.ipynb`).
Format: `[b0, b1, b2, -a1, -a2]` per section — note the **negated** $a$ coefficients,
matching the firmware's Direct Form II Transposed convention.

In [ ]:
# High-pass Chebyshev I, 200 Hz, order 4 (2 biquad sections)
# [b0, b1, b2, -a1, -a2]  — verbatim from dsp-core/src/lib.rs HIGHPASS_COEFFS
HIGHPASS_COEFFS = np.array([
    [8.8843882084e-01, -1.7768776417e+00, 8.8843882084e-01, 1.8572715521e+00, -8.6588841677e-01],
    [1.0000000000e+00, -2.0000000000e+00, 1.0000000000e+00, 1.9743645191e+00, -9.7780704498e-01],
], dtype=np.float32)

# Low-pass Chebyshev I, 2800 Hz, order 4 (2 biquad sections)
# [b0, b1, b2, -a1, -a2]  — verbatim from dsp-core/src/lib.rs LOWPASS_COEFFS
LOWPASS_COEFFS = np.array([
    [1.0223765858e-02, 2.0447531715e-02, 1.0223765858e-02, 1.1647846699e+00, -4.0787613392e-01],
    [1.0000000000e+00, 2.0000000000e+00, 1.0000000000e+00, 1.0389341116e+00, -7.3549830914e-01],
], dtype=np.float32)

## Compressor parameters

Copied verbatim from `src/dsp.rs`.

In [ ]:
COMP_THRESHOLD   = 0.3
COMP_RATIO       = 4.0
COMP_MAKEUP_GAIN = 1.5

# 1 - exp(-1 / (FS * attack_time_s))  where attack_time = 0.0002 s
COMP_ATTACK_COEFF  = 0.22119922
# 1 - exp(-1 / (FS * release_time_s)) where release_time = 0.1 s
COMP_RELEASE_COEFF = 0.00049987503

## Helper: Direct Form II Transposed biquad cascade

Implements the exact same update loop as `BiquadState::process()` in the firmware:

$$y = b_0 x + s_0$$
$$s_0 \leftarrow b_1 x + (-a_1) y + s_1$$
$$s_1 \leftarrow b_2 x + (-a_2) y$$

In [ ]:
def biquad_cascade(coeffs: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Apply a cascade of N biquad sections to signal x.

    coeffs : (N, 5) array, each row = [b0, b1, b2, -a1, -a2]
    x      : 1-D float32 signal
    Returns: filtered signal, same length as x
    """
    y = x.copy().astype(np.float32)
    for b0, b1, b2, na1, na2 in coeffs:
        s0, s1 = 0.0, 0.0
        out = np.empty_like(y)
        for i, xin in enumerate(y):
            yout   = b0 * xin + s0
            s0     = b1 * xin + na1 * yout + s1
            s1     = b2 * xin + na2 * yout
            out[i] = yout
        y = out
    return y

## Helper: voice compressor

Mirrors `Compressor::process()` in the firmware: envelope follower with asymmetric
attack/release, gain computed from overshoot above threshold.

In [ ]:
def compress(x: np.ndarray) -> np.ndarray:
    """Apply the firmware voice compressor to signal x (float32, normalised ±1)."""
    envelope = 0.0
    gain     = 1.0
    out      = np.empty_like(x)
    for i, sample in enumerate(x):
        abs_s = abs(float(sample))
        if abs_s > envelope:
            envelope += COMP_ATTACK_COEFF  * (abs_s - envelope)
        else:
            envelope += COMP_RELEASE_COEFF * (abs_s - envelope)

        if envelope > COMP_THRESHOLD:
            overshoot  = envelope / COMP_THRESHOLD
            compressed = 1.0 + (overshoot - 1.0) / COMP_RATIO
            gain       = COMP_THRESHOLD * compressed / envelope
        else:
            gain = 1.0

        out[i] = np.clip(sample * gain * COMP_MAKEUP_GAIN, -1.0, 1.0)
    return out.astype(np.float32)

## Load and resample input

In [ ]:
src_rate, data = wavfile.read(INPUT_WAV)

# Convert to float32 in [-1, 1]
if data.dtype == np.int16:
    audio = data.astype(np.float32) / 32768.0
elif data.dtype == np.int32:
    audio = data.astype(np.float32) / 2**31
elif data.dtype == np.float32:
    audio = data
else:
    audio = data.astype(np.float32)

# Mix down to mono if stereo
if audio.ndim == 2:
    audio = audio.mean(axis=1)

# Resample to FS if needed
if src_rate != FS:
    n_samples = int(len(audio) * FS / src_rate)
    audio = signal.resample(audio, n_samples).astype(np.float32)
    print(f'Resampled {src_rate} Hz → {FS} Hz  ({len(audio)} samples)')
else:
    print(f'Input already at {FS} Hz  ({len(audio)} samples)')

duration_s = len(audio) / FS
print(f'Duration: {duration_s:.2f} s')

## Run the audio chain

In [ ]:
after_hp         = biquad_cascade(HIGHPASS_COEFFS, audio)
after_compressor = compress(after_hp)
after_lp         = biquad_cascade(LOWPASS_COEFFS, after_compressor)

# Save output WAV (int16, 20 kHz)
out_int16 = np.clip(after_lp, -1.0, 1.0)
out_int16 = (out_int16 * 32767).astype(np.int16)
wavfile.write(OUTPUT_WAV, FS, out_int16)
print(f'Written: {OUTPUT_WAV}')

## Spectrograms

In [ ]:
stages = [
    ('Input',           audio),
    ('After HP 200 Hz', after_hp),
    ('After compressor',after_compressor),
    ('After LP 2800 Hz',after_lp),
]

fig, axes = plt.subplots(len(stages), 1, figsize=(12, 3 * len(stages)), sharex=True)
for ax, (title, sig) in zip(axes, stages):
    ax.specgram(sig, Fs=FS, NFFT=512, noverlap=384, cmap='inferno', vmin=-80)
    ax.set_title(title)
    ax.set_ylabel('Frequency (Hz)')
    ax.set_ylim(0, FS / 2)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## Waveform overview

In [ ]:
t = np.arange(len(audio)) / FS

fig, axes = plt.subplots(len(stages), 1, figsize=(12, 2 * len(stages)), sharex=True)
for ax, (title, sig) in zip(axes, stages):
    ax.plot(t, sig, linewidth=0.4)
    ax.set_title(title)
    ax.set_ylabel('Amplitude')
    ax.set_ylim(-1.1, 1.1)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()